In [2]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="VjAxZSEJru3p0Pij8zYu")
project = rf.workspace("mehdis-workspace-jupje").project("football-analyst-v2-v5tiv")
version = project.version(2)
dataset = version.download("yolov8")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Football-Analyst-v2-2 in yolov8:: 100%|██████████| 16917/16917 [00:06<00:00, 2700.40it/s]


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import yaml

# Lire le fichier généré par Roboflow
with open(f"{dataset.location}/data.yaml", 'r') as stream:
    data = yaml.safe_load(stream)
    print(data)
# print("Nombre de classes (nc) :", data['nc'])
# print("Noms des classes :", data['names'])

{'names': ['Ball', 'Keeper', 'Player', 'Ref'], 'nc': 4, 'roboflow': {'license': 'CC BY 4.0', 'project': 'football-analyst-v2-v5tiv', 'url': 'https://universe.roboflow.com/mehdis-workspace-jupje/football-analyst-v2-v5tiv/dataset/2', 'version': 2, 'workspace': 'mehdis-workspace-jupje'}, 'test': '../test/images', 'train': '../train/images', 'val': '../valid/images'}


In [5]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 31.1 MB/s eta 0:00:00


In [2]:
# Exemple de ce que vous feriez juste après :
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/best.pt")  # Charge un modèle YOLOv8 natif
model.train(data=f"{dataset.location}/data.yaml", epochs=100)  # Lance l'entraînement !

NameError: name 'dataset' is not defined

In [3]:
model = YOLO(r"runs\detect\train\weights\best.pt")
resultat  = model("../frame_1.jpg")
resultat[0].show()


image 1/1 D:\MEHDI\Study\WISD\S2\Visual analytics\projet\Analyse-Tactique-du-Football-par-Vision-par-Ordinateur\notebooks\fine_tunning_3\..\frame_1.jpg: 384x640 22 Players, 5 Refs, 214.6ms
Speed: 11.6ms preprocess, 214.6ms inference, 4.6ms postprocess per image at shape (1, 3, 384, 640)


In [9]:
# 2. Lancer la validation exclusivement sur le jeu de test
metrics = model.val(split='val')

# 3. Afficher les résultats clés
print(f"mAP50-95 (Précision globale) : {metrics.box.map:.4f}")
print(f"mAP50 (Précision à seuil 0.5) : {metrics.box.map50:.4f}")
print(f"Précision par classe : {metrics.box.mp:.4f}")
print(f"Rappel (Recall) par classe : {metrics.box.mr:.4f}")

# Afficher les performances détaillées par classe
for i, name in enumerate(metrics.names.values()):
    print(f"--- Classe : {name} ---")
    print(f"  Précision (Precision) : {metrics.box.class_result(i)[0]:.4f}")
    print(f"  Rappel (Recall)       : {metrics.box.class_result(i)[1]:.4f}")
    print(f"  mAP50                 : {metrics.box.class_result(i)[2]:.4f}")
    print("-" * 25)

Ultralytics 8.4.88 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1825.4±708.5 MB/s, size: 157.7 KB)
val: Scanning /content/Football-Analyst-v2-2/valid/labels.cache... 560 images, 35 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 560/560 130.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 35/35 3.1it/s 11.4s
                   all        560       7173      0.879      0.619      0.688      0.417
                  Ball        275        299      0.833      0.551      0.625       0.35
                Keeper        247        255      0.883      0.569      0.672      0.395
                Player        399       5963      0.931      0.717      0.747      0.494
                   Ref        353        656      0.867      0.639      0.709      0.429
Speed: 1.9ms preprocess, 4.4ms inference, 0.0ms loss, 3.1ms postprocess per image
Results saved to /content/run

In [4]:
import cv2
from ultralytics import YOLO

model = YOLO(r"runs\detect\train\weights\best.pt")
cap = cv2.VideoCapture(r"..\video1.mp4")

# Récupérer les propriétés de la vidéo d'origine
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Créer le fichier de sortie
out = cv2.VideoWriter(r".\resultat_annote1.mp4", cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    resultats = model(frame)
    frame_annotee = resultats[0].plot()

    out.write(frame_annotee)        # écrit la frame dans le fichier vidéo
    cv2.imshow("Detection", frame_annotee)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
out.release()
cv2.destroyAllWindows()


0: 384x640 1 Keeper, 19 Players, 147.1ms
Speed: 8.1ms preprocess, 147.1ms inference, 2.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Keeper, 19 Players, 148.0ms
Speed: 3.7ms preprocess, 148.0ms inference, 2.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Keeper, 19 Players, 139.6ms
Speed: 6.4ms preprocess, 139.6ms inference, 2.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Keeper, 18 Players, 175.2ms
Speed: 4.8ms preprocess, 175.2ms inference, 2.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Keeper, 18 Players, 143.2ms
Speed: 5.3ms preprocess, 143.2ms inference, 2.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Keeper, 18 Players, 178.2ms
Speed: 2.9ms preprocess, 178.2ms inference, 2.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Keeper, 18 Players, 1 Ref, 158.8ms
Speed: 5.1ms preprocess, 158.8ms inference, 2.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1